# 060. Transformacje, Grupowanie i Łączenie Danych

Zrozumienie parametru `axis`, tabele przestawne `pivot_table` i `melt`, grupowanie `groupby` w stylu SQL (`as_index=False`, `NamedAgg`, `transform`) oraz defensywne złączenia `pd.merge` z `validate='1:m'` i `indicator=True`.


## 1. Zrozumienie osi `axis=0` vs `axis=1`


In [ ]:
import pandas as pd
import numpy as np


In [ ]:

oceny = pd.DataFrame({
    'Matematyka': [4, 5, 3],
    'Fizyka': [3, 4, 4],
    'Informatyka': [5, 5, 4]
}, index=['Uczeń_1', 'Uczeń_2', 'Uczeń_3'])


In [ ]:

# axis=0: pionowo w dół (średnia z każdego przedmiotu)
print("Średnia per przedmiot (axis=0):\n", oceny.mean(axis=0))


In [ ]:

# axis=1: poziomo w poprzek (średnia każdego ucznia)
print("Średnia per uczeń (axis=1):\n", oceny.mean(axis=1))


## 2. Zmiana kształtu: `pivot_table()` vs `melt()`


In [ ]:
df_sprzedaz = pd.DataFrame({
    'Region': ['Północ', 'Południe', 'Północ', 'Południe'],
    'Produkt': ['Laptop', 'Laptop', 'Mysz', 'Laptop'],
    'Sprzedaz': [4000, 4200, 150, 4100]
})

# Tabela przestawna (Format Szeroki)
tabela_p = df_sprzedaz.pivot_table(
    index='Region',
    columns='Produkt',
    values='Sprzedaz',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='SUMA'
)
print("Tabela przestawna:")
print(tabela_p)


## 3. Groupby w stylu SQL (`as_index=False`) z `NamedAgg`


In [ ]:
df_kadry = pd.DataFrame({
    'Dzial': ['IT', 'IT', 'IT', 'Marketing', 'Marketing', 'Finanse'],
    'Pracownik': ['Jan', 'Anna', 'Tomasz', 'Ewa', 'Piotr', 'Zofia'],
    'Pensja': [16000, 12000, 8000, 11000, 7500, 18000],
    'Staz': [5, 3, 1, 4, 2, 8]
})

# Agregacja SQL-style z as_index=False i czytelnymi nazwami kolumn:
raport_plac = df_kadry.groupby('Dzial', as_index=False).agg(
    Liczba_Pracownikow=('Pensja', 'count'),
    Srednia_Pensja=('Pensja', 'mean'),
    Max_Pensja=('Pensja', 'max')
)
print("Raport płac:")
print(raport_plac)


## 4. `transform()` – udział w grupie bez zwijania wierszy


In [ ]:
# transform zachowuje oryginalną liczbę wierszy:
budzet_dzialu = df_kadry.groupby('Dzial')['Pensja'].transform('sum')
df_kadry['Udzial_w_Budzecie'] = (df_kadry['Pensja'] / budzet_dzialu).round(3)
print(df_kadry[['Dzial', 'Pracownik', 'Pensja', 'Udzial_w_Budzecie']])


## 5. Defensywne złączenia `pd.merge()` z `validate` i `indicator`


In [ ]:
klienci = pd.DataFrame({'id': [1, 2, 3, 4], 'nazwisko': ['Kowalski', 'Nowak', 'Wiśniewski', 'Zielińska']})
zamowienia = pd.DataFrame({'id_zam': [101, 102], 'id': [1, 2], 'kwota': [250.0, 490.0]})

# validate='1:m' chroni przed powieleniem wierszy przez duplikaty
polaczone = pd.merge(klienci, zamowienia, on='id', how='left', validate='1:m', indicator=True)
print("Klienci bez zamówień (_merge == 'left_only'):")
print(polaczone.query("_merge == 'left_only'"))
